# Key recovery desde diferenciales ya calculados

Este notebook ejecuta recuperacion de clave para SPN16, AES reducido y KLEIN reducido usando diferenciales conocidos. No vuelve a buscar diferenciales; solo usa `Delta_in` y, para AES/KLEIN, `Delta` en la penultima ronda.

In [ ]:
from pathlib import Path
import sys


def add_repo_root_to_path():
    current = Path.cwd().resolve()
    for path in [current, *current.parents]:
        if (path / "cryptosystems").is_dir() and (path / "differential_cryptoanalysis").is_dir():
            if str(path) not in sys.path:
                sys.path.insert(0, str(path))
            return path
    raise RuntimeError("No se encontro la raiz del repositorio")


ROOT = add_repo_root_to_path()
print(f"Repo root: {ROOT}")

from differential_cryptoanalysis.key_recovery_from_differentials import (
    print_aes_result,
    print_klein_result,
    print_spn16_result,
    recover_reduced_aes_from_differential,
    recover_reduced_klein_from_differential,
    recover_spn16_from_differential,
)

## SPN16

Aqui si se generan pares elegidos desde textos claros: `P` y `P ^ Delta_in`. El diccionario `expected_delta_u_by_nibble` usa posiciones de nibble `0..3`, donde `0` es el nibble mas significativo.

In [ ]:
spn16_result = recover_spn16_from_differential(
    master_key_hex="00112233445566778899AABB",
    rounds=5,
    delta_in=0x0B00,
    expected_delta_u_by_nibble={2: 0x5},
    n_pairs=4000,
    seed=2026,
)

print_spn16_result(spn16_result, top=5)

## AES reducido

Para AES reducido se usan pares correctos sinteticos en la entrada de la ultima ronda. Esto permite probar la fase de recuperacion de `K_r` usando el diferencial de la penultima ronda que ya fue calculado.

In [ ]:
aes_result = recover_reduced_aes_from_differential(
    master_key_hex="0011223344556677",
    rounds=4,
    block_bits=64,
    key_bits=64,
    delta_in=0x1800000000000000,
    delta_penultimate=0xB8D172C63DC37207,
    n_pairs=64,
    seed=2026,
)

print_aes_result(aes_result, top=5)

## KLEIN reducido

Para KLEIN reducido se recuperan nibbles activos de `T = InvRotate(InvMix(K_(r+1)))`. Los nibbles no activos quedan como `?`; si el espacio es pequeno, se enumeran candidatos de `K_(r+1)`.

In [ ]:
klein_result = recover_reduced_klein_from_differential(
    master_key_hex="0011223344556677",
    rounds=6,
    block_bits=32,
    key_bits=64,
    delta_in=0x10000000,
    delta_penultimate=0xF07BCBB0,
    n_pairs=128,
    seed=2026,
)

print_klein_result(klein_result, top=5)

## Como cambiar el diferencial

- SPN16: cambia `delta_in` y `expected_delta_u_by_nibble`.
- AES/KLEIN: cambia `delta_in` solo para documentar el trail y `delta_penultimate` para la recuperacion real de la ultima ronda.
- Aumenta `n_pairs` si hay ruido o si el ranking de candidatos no separa bien al correcto.